In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from utils import targetEncode, labelEncode, oneHotEncode

In [3]:
engData = pd.read_csv('./data/eng_data.csv')
print(engData.head().to_markdown())

|    | Timestamp           |   TransactionID |   AccountID |   Amount | Merchant   | TransactionType   | Location    |   Year |   Quarter |   Month | Day_Of_Week   |   Day_Of_Month |   Day_Of_Year |   Hour |   minute |   Minute |
|---:|:--------------------|----------------:|------------:|---------:|:-----------|:------------------|:------------|-------:|----------:|--------:|:--------------|---------------:|--------------:|-------:|---------:|---------:|
|  0 | 2023-01-01 08:00:00 |            1127 |           4 | 95071.9  | H          | Purchase          | Tokyo       |   2023 |         1 |       1 | Sunday        |              1 |             1 |      8 |        0 |        0 |
|  1 | 2023-01-01 08:01:00 |            1639 |          10 | 15607.9  | H          | Purchase          | London      |   2023 |         1 |       1 | Sunday        |              1 |             1 |      8 |        1 |        1 |
|  2 | 2023-01-01 08:02:00 |             872 |           8 | 65092.3  | E       

In [9]:

preppedData = engData.drop(columns=['Timestamp', 'TransactionID', 'AccountID'])
preppedData = targetEncode(data=preppedData, encodeFeatures=['TransactionType', 'Location', 'Day_Of_Week', 'Merchant'], targetFeature='Amount')
preppedData['Membership'] = None
preppedData['C1'] = None
preppedData['C2'] = None
preppedData['C3'] = None
preppedData['C4'] = None
print(preppedData.head().to_markdown())


|    |   Amount |   Merchant |   TransactionType |   Location |   Year |   Quarter |   Month |   Day_Of_Week |   Day_Of_Month |   Day_Of_Year |   Hour |   minute |   Minute | Membership   | C1   | C2   | C3   | C4   |
|---:|---------:|-----------:|------------------:|-----------:|-------:|----------:|--------:|--------------:|---------------:|--------------:|-------:|---------:|---------:|:-------------|:-----|:-----|:-----|:-----|
|  0 | 95071.9  |    50455.3 |           50243.5 |    50393.6 |   2023 |         1 |       1 |       50165.7 |              1 |             1 |      8 |        0 |        0 |              |      |      |      |      |
|  1 | 15607.9  |    50455.3 |           50243.5 |    50241.7 |   2023 |         1 |       1 |       50165.7 |              1 |             1 |      8 |        1 |        1 |              |      |      |      |      |
|  2 | 65092.3  |    50217.7 |           50141.6 |    50241.7 |   2023 |         1 |       1 |       50165.7 |              1 | 

In [ ]:
# Clustering Functions
def FCM(row, clusters, p=2, basin = False, features: list[str] = ['observed']):
    alpha = 0.000001
    sumSimm = 0
    clusterLabels = list(clusters.keys())
    for name, values in clusters.items():
        x = [row[feature] for feature in features]
        
        diff = [a-b for a,b in zip(x,values['center'])]
        clusterDist = pow(sum([pow(d,p) for d in diff]),1/p)
        clusterSimm = (1/(clusterDist + alpha)) * (values['weight'] if basin else 1)

        row[name] = clusterSimm
        sumSimm += clusterSimm

    for name in clusterLabels:
        row[name] = row[name]/sumSimm

    row['membership'] = row[clusterLabels].idxmax()
    return row
    
def cluster(input_matrix, initialClusters, membershipFunc, center = 'mean', basin = False, features: list[str] = ['observed']):
    alpha = 0.0000003
    Clusters = initialClusters.copy()

    membership_matrix = input_matrix.copy(deep=True)
    populationCount = len(membership_matrix)
    
    iterationCounter = 1
    changeDetected = True
    while((changeDetected) and (iterationCounter < 30)):
        print('iteration:', iterationCounter)
        iterationCounter += 1
        
        # Assign Cluster Membership
        membership_matrix = membership_matrix.apply(membershipFunc, axis=1, clusters=Clusters, basin=basin, features=features)
        changeDetected = False
        for name, group in membership_matrix.groupby('membership'):
            groupPop = group[name].count()
            clusterCenter = [group[feature].mean() if center=='mean' else group[feature].median() for feature in features]
            clusterWeight = groupPop / populationCount
            clusterCoviariance = group[features].cov() if len(features) > 1 else group[features].var()
            
            if ((Clusters[name]['weight'] != clusterWeight) or (any([a != b for a,b in zip(Clusters[name]['center'], clusterCenter)])) or (any([a != b for a,b in zip(Clusters[name]['cov'], clusterCoviariance)]))):
                changeDetected = True
            
            Clusters[name] = {
                'center': clusterCenter,
                'weight': clusterWeight + alpha,
                'cov': clusterCoviariance
            }


    return [membership_matrix, Clusters]

In [ ]:
print(preppedData.head().to_markdown())

clusterFeatures = ['Amount', 'Merchant', 'TransactionType', 'Location', 'Quarter', 'Month', 'Day_Of_Week','Day_Of_Month','Day_Of_Year','Hour','Minute']
initial_clusters = {
    'C1':{
        'center': [1,1,1,1,1,1,1,1,1,1,1]
    },
    'C2':{
        'center': [4000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000]
    },
    'C3':{
        'center': [6000,2000,2000,3000,0,0,0,0,0,0,0,0]
    },
    'C4':{
        'center': [100000,3000,5000,5000,0,0,0,0,0,0,0]
    }
}
mem_matrix, clusters = cluster(preppedData, )

|    |   Amount |   Merchant |   TransactionType |   Location |   Year |   Quarter |   Month |   Day_Of_Week |   Day_Of_Month |   Day_Of_Year |   Hour |   minute |   Minute | Membership   | C1   | C2   | C3   | C4   |
|---:|---------:|-----------:|------------------:|-----------:|-------:|----------:|--------:|--------------:|---------------:|--------------:|-------:|---------:|---------:|:-------------|:-----|:-----|:-----|:-----|
|  0 | 95071.9  |    50455.3 |           50243.5 |    50393.6 |   2023 |         1 |       1 |       50165.7 |              1 |             1 |      8 |        0 |        0 |              |      |      |      |      |
|  1 | 15607.9  |    50455.3 |           50243.5 |    50241.7 |   2023 |         1 |       1 |       50165.7 |              1 |             1 |      8 |        1 |        1 |              |      |      |      |      |
|  2 | 65092.3  |    50217.7 |           50141.6 |    50241.7 |   2023 |         1 |       1 |       50165.7 |              1 | 